### **DimUser**

In [0]:

import uuid
# This function is used to preview the data in a stream
def preview(df, rows=10):
    unique_id = str(uuid.uuid4())[:8]
    temp_path = f"abfss://silver@umutcanasci.dfs.core.windows.net/_temp_checkpoints/{unique_id}"
    table_name = f"temp_preview_{unique_id}"
    
    query = df.writeStream \
      .format("memory") \
      .queryName(table_name) \
      .outputMode("append") \
      .option("checkpointLocation", temp_path) \
      .trigger(availableNow=True) \
      .start()
    
    query.awaitTermination()
    display(spark.sql(f"SELECT * FROM {table_name} LIMIT {rows}"))

In [0]:
# DBTITLE 1,Import Required Libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *

# add the util folder to the python path
import os
import sys
project_pth = os.path.join(os.getcwd(), "..","..")
sys.path.append(project_pth)

from utils.transformations import reusable

### **AUTO LOADER** 

In [0]:
df_user = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/checkpoint")\
    .load("abfss://bronze@umutcanasci.dfs.core.windows.net/DimUser")

In [0]:
display(
    df_user, 
    checkpointLocation="abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/temp_display/cell_9_2"
)

In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))
display(
    df_user,
    checkpointLocation="abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/temp_display/cell_7"
)

In [0]:
# /Workspace/Users/umutcanasci@posta.mu.edu.tr/spotify_dab/utilities/transformations.py

In [0]:
spark.sql("DROP TABLE IF EXISTS azureproject_catalog.silver.dimUser_table")
dbutils.fs.rm("abfss://silver@umutcanasci.dfs.core.windows.net/dimUser", recurse=True)

In [0]:
import os
import sys

project_pth = os.path.join(os.getcwd(), "..", "..")
if project_pth not in sys.path:
    sys.path.append(project_pth)

from pyspark.sql.functions import col, upper
from utils.transformations import reusable

In [0]:

# Bronze katmanından User verisini oku
df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/checkpoint") \
    .load("abfss://bronze@umutcanasci.dfs.core.windows.net/DimUser")

# Dönüşümler
df_user = df_user.withColumn("user_name", upper(col("user_name")))
df_user = reusable().dropColumns(df_user, ["_rescued_data"])
df_user = df_user.dropDuplicates(["user_id"])

# Silver tablosuna yaz
df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/checkpoint") \
    .trigger(once=True) \
    .option("path", "abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/data") \
    .toTable("azureproject_catalog.silver.dimUser_table")

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user, ["_rescued_data"])
df_user = df_user.dropDuplicates(["user_id"])

display(
    df_user,
    checkpointLocation="abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/temp_display/cell_10_2"
)

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@umutcanasci.dfs.core.windows.net/dimUser/data")\
    .toTable("azureproject_catalog.silver.dimUser_table")


##   **dimArtist** 

In [0]:
df_art = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimArt/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@umutcanasci.dfs.core.windows.net/DimArtist")


In [0]:
display(
    df_art,
    checkpointLocation="abfss://silver@umutcanasci.dfs.core.windows.net/dimArt/temp_display/cell_12_2"
)


In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art, ["_rescued_data"])
df_art = df_art.dropDuplicates(["artist_id"])

display(
    df_art,
    checkpointLocation="abfss://silver@umutcanasci.dfs.core.windows.net/dimArt/temp_display/cell_13_2"
)

In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimArt/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@umutcanasci.dfs.core.windows.net/dimArt/data")\
    .toTable("azureproject_catalog.silver.dimArt_table")

In [0]:
# check the data in silver
df_silver_check = spark.read.format("delta").load("abfss://silver@umutcanasci.dfs.core.windows.net/dimArt/data")
display(df_silver_check)

## **dimTrack**

In [0]:
df_track  = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimTrack/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@umutcanasci.dfs.core.windows.net/DimTrack")

display(
    df_track,
    checkpointLocation="abfss://silver@umutcanasci.dfs.core.windows.net/dimTrack/temp_display/cell_16_4"
)


In [0]:
from pyspark.sql.functions import when, col, regexp_replace
 
df_track = df_track.withColumn("duration_flag", when(col("duration_sec") < 150, "low")\
                               .when(col("duration_sec") < 300, "medium")\
                               .otherwise("high"))

df_track = df_track.withColumn("track_name",regexp_replace(col("track_name"), "-", " "))

df_track = reusable().dropColumns(df_track, ["_rescued_data"])


In [0]:
preview(df_track)


In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@umutcanasci.dfs.core.windows.net/dimTrack/data")\
    .toTable("azureproject_catalog.silver.dimTrack_table")
#check the data in silver
df_silver_check = spark.read.format("delta").load("abfss://silver@umutcanasci.dfs.core.windows.net/dimTrack/data")
display(df_silver_check)

## **dimDate**

In [0]:
df_date  = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimDate/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@umutcanasci.dfs.core.windows.net/DimDate")

df_date = reusable().dropColumns(df_date, ["_rescued_data"])

In [0]:
preview(df_date)

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/dimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@umutcanasci.dfs.core.windows.net/dimDate/data")\
    .toTable("azureproject_catalog.silver.dimDate_table")
#check the data in silver
df_silver_check = spark.read.format("delta").load("abfss://silver@umutcanasci.dfs.core.windows.net/dimDate/data")
display(df_silver_check)

## **factStream**

In [0]:
df_factStream = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/factStream/checkpoint")\
    .option("schemaEvolutionMode", "addNewColumns")\
    .load("abfss://bronze@umutcanasci.dfs.core.windows.net/FactStream")



In [0]:
preview(df_factStream)

In [0]:
df_factStream = reusable().dropColumns(df_factStream, ["_rescued_data"])
df_factStream.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@umutcanasci.dfs.core.windows.net/factStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@umutcanasci.dfs.core.windows.net/factStream/data")\
    .toTable("azureproject_catalog.silver.factStream_table")


In [0]:
display(spark.read.format("delta").load("abfss://silver@umutcanasci.dfs.core.windows.net/factStream/data"))